# Title: Enhancing Large Vision-Language Conversational (LaViC) Recommendation Framework

#### Members' Names: Jason Yu, Jessie Ma, Yosef Moustafa

####  Emails: m10yu@torontomu.ca, qian.ma@torontomu.ca, ymoustafa@torontomu.ca

# Introduction:

#### Problem Description:

Group 8’s focus in this project is to investigate the impact of visual data density on the performance of LaViC (Large Vision-Language Conversational) Recommendation Framework proposed by Jeon et al. LaViC is a two-step process that uses LLaVA v1.6 (Mistral-7b) as the Large Language Model (LLM). The steps are: 1) Visual knowledge self-distillation and 2) Recommendation fine-tuning [3]. The paper leveraged an original dataset created by the researchers consisting of three Amazon product categories: 1) Beauty, 2) Fashion, and 3) Home. The aim is to understand how varying the number of images used during training in the visual self-distillation stage affects the quality of knowledge distillation within the Amazon Home category. The project compares a text-only baseline of using solely product titles from Amazon against 3 distinct experimental trials that gradually increase in image sample size. The trials included a single-image distillation (1 image shown per item), dual-image distillation (2 images per item), and a complete-image set distillation of the Home category (all images). By maintaining a consistent batch size of 2 and epoch number of 2 across all 3 trials, the study is able to effectively evaluate whether the number of image inputs during the visual distillation process has any impact on the performance of the framework. Understanding the effect this scalability has on the model's performance can reduce heavy computational overhead associated with processing a high volume of images.

#### Context of the Problem:

Large Vision-Language Models (VLMs) such as LLaVA v1.6 (Mistral-7b) have shown remarkable capabilities in understanding multimodal data. However in some specialized domains that rely on visual characteristics, such as e-commerce domains, particularly in fashion, home decor and the likes, relying on text-based inputs could potentially lose important information found in visual representations of the product. Missing out on these visual features may cause the model to provide recommendations that are not consistent with a seeker’s preferences. These VLMs also typically employ a large-scale vision encoder such as CLIP or SigLIP [3] which convert images into thousands of visual tokens leading to exponential growth in sequence length. Exponentially large sequence lengths cause the token explosion problem which slows down inference and consumes immense GPU memory especially when multiple images are fed to the model to be analyzed and processed. The LaViC (Large Vision-Language Conversational) Recommendation Framework proposed by Jeon et al. aims to bridge these gaps by compressing a large model such as LLaVA into a more efficient framework which concurrently also extracts more meaning from the combination of both textual and visual inputs.

#### Limitation About other Approaches:

Prior methods relied primarily on unimodal inputs such as text-based embeddings or visual embeddings which disregard semantic relationships that can be captured between both textual and visual information. On the other hand, multimodal approaches face a token overflow or explosion problem. 

#### Solution:

LaViC addresses both aforementioned limitations by using a self-distillation mechanism which condenses thousands of raw visual tokens into a small set of manageable ‘summary’ tokens called CLS tokens. The framework distills an image into only 5 CLS tokens, so that at the recommendation fine-tuning stage when the LLM is fed with 10 image candidates to choose from to recommend to the seeker, only 50 (5x50) CLS tokens are required whereas the traditional LLaVA VLM would need 28,850 (5x577x10) tokens, far exceeding the 4,096 token context limit [3]. As a result, this significantly reduces computational cost and enables more efficient training and inference at both the distillation and recommendation stages.

# Background

| Reference |Explanation |  Dataset/Input |Weakness
| --- | --- | --- | --- |
| Liu et al. [1] | Instead of visual prompting, they converted each product image into a text summary using the VLM's own visual understanding and used these summaries alongside the user history to rank candidate items for recommendation.| Amazon Review datasets (Sports, Clothing, Beauty, Toys) | As a zero-shot approach with no fine-tuning, it lacks domain-specific knowledge and struggles to effectively handle multiple product images simultaneously.
| Wei et al. [2] | Arguing that conversation context alone is too short to capture user preferences in a conversational recommender system, they extracted both text and image features of items mentioned in conversations, built separate graph structures for each modality. These graphs were then combined with prompt learning to guide an LLM to make better recommendations| ReDial and INSPIRED (movie conversation datasets) | Relies on a complex pipeline of separate, smaller models rather than a unified VLM, and is limited to movie domains with no generalization to other domains.
| Jeon et al. [3] | They proposed a two-stage framework where, first, a VLM is trained to compress product images from hundreds of tokens down to just a few visual tokens (self-distillation), then the model is fine-tuned to combine those compressed image tokens with conversation history to select the correct item from candidates.| Reddit-Amazon (fashion, beauty, home) | Currently evaluated on a limited set of visually-driven domains (fashion, beauty, and home), with scaling to broader domains left as future work.


# Methodology

### Figure 1: Stage 1 - LaViC Visual Knowledge Self-Distillation Process
![Vision Distillation Diagram](images/distillation.png)
#### This image shows the first stage of LaViC and highlights two methodologies of generating a textual description from an image and prompt input. A transformer distills the image into sub-image tokens (vision tower) and maps them to LLM-aligned embeddings to feed into the LLM as the next step (projector). The vision tower and projector consitute the vision module. The left side demonstrates the generation methodology of a traditional VLM such as LLaVA which generates thousands of sub-image tokens from one image, leading to the token explosion problem discussed earlier. The right side demonstrates the LaVic methodology of distillation which freezes the LLM and applies LoRA on the vision module to produce only 5 CLS tokens, greatly reducing computation [3]. As you can see, although LaViC uses fewer tokens per image, it is equally able to generate the same product description as the traditional method.

### Figure 2: Stage 2 - LaViC Recommendation Fine-Tuning Process
![Recommendation Fine Tuning Diagram](images/recommendation.png)
#### This image shows the second stage of LaViC, where the output from stage 1 is used to train the LLM. Here, we freeze the vision module parameters and apply LoRA on the LLM instead to produce 10 candidates of images along with their item ID and Amazon titles, and the model chooses the ground-truth item. There is only one ground-truth item per request/textual description.

We focused on one method proposed in the paper to improve item recommendation with image data. Due to high computational cost and financial constraints, we restricted our experiments to the Amazon Home category, rather than using all 3 categories (Beauty, Fashion, and Home) in the original paper.

Each item in our dataset consists of 1–9 images capturing different angles and perspectives. To study how the amount of visual information affects model performance, we constructed 3 sub-training sets using 1, 2, and all available images per item. For each sub-training set, we followed the pipeline proposed in the paper. Specifically, we first performed visual knowledge self-distillation to obtain compact visual tokens from each image, and then conducted recommendation fine-tuning to integrate dialogue context with the distilled visual tokens for candidate-based recommendation. By keeping the pipeline fixed and varying only the number of images, we are able to isolate and evaluate the contribution of richer visual information to the final recommendation performance.

This main pipeline was implemented and executed on NVIDIA A100 GPU instances via Google Colab, which provided sufficient computational resources to run the full method as described.

We used LLaVA-v1.6 as the backbone model for this project. For the baseline, we input only the item title and allow the model to select from candidate items. Performance is evaluated using recall and candidate validity. For the LaViC-based method, both image and text inputs are provided to the fine-tuned model, and the same evaluation metrics are used for comparison.

In addition, we implemented a resource-constrained variant of the pipeline on a Kaggle environment using a NVIDIA T4 GPU with 16GB of VRAM. Due to hardware limitations, several modifications were introduced: we applied 4-bit NF4 QLoRA quantization to reduce memory usage, truncated candidate lists to the top 5 items, and limited the dialogue context to 300 characters.

We also replaced the original model class with LlavaNextForConditionalGeneration in the Kaggle implementation, which corresponds to LLaVA-NeXT (v1.6) and improves upon earlier versions through higher image resolution and better visual instruction tuning. To ensure training stability within Kaggle session limits, we implemented a custom checkpointing mechanism that saves LoRA weights every 500 steps.

However, it is computationally infeasible to construct a stronger baseline that includes both image and text inputs using LLaVA-v1.6. Converting images into tokens leads to a significant increase in input length, resulting in extremely high inference costs as discussed in the report section on limitations. As noted in the original paper, such a baseline would require multiple days of runtime even with dual NVIDIA A100 GPUs. Therefore, this baseline is excluded from our experiments.

# Implementation

Due to the length of two main .py files, we will showcase only some core functions here:

### Visual Knowledge Self-Distillation

This is the first step of the two-part pipeline.  On a high level the goal is to compresses each image into a few compact (5) tokens while keeping the same visual meaning.  It outputs a trained LoRA vision adapter that compresses each image into a few CLS embeddings.  No predication is done here.

In [2]:
# ---------------------------------------------------------
# 1) Special tokens (5 sub-image placeholders)
# ---------------------------------------------------------
IMAGE_TOKENS = [
    "<ItemImageEmb1>", "<ItemImageEmb2>", "<ItemImageEmb3>",
    "<ItemImageEmb4>", "<ItemImageEmb5>"
]

# ---------------------------------------------------------
# 2) Prompt Template
# ---------------------------------------------------------
PROMPT_TEMPLATE = (
    "You are a helpful assistant.\n"
    "Given an Amazon product's title and its image, please provide a detailed, visually grounded description of the product "
    "that would help someone decide whether to purchase it. "
    "Focus on the product's appearance, features, and any other visually informative aspects. "
    "Do not mention the product's title in your answer. "
    "This product's title is: {title}\n"
    f"{''.join(IMAGE_TOKENS)}\n\n"
    "Assistant:"
)




Below is the PretrainVisionModel class, it trains LoRA adapters on a vision-language model's vision encoder and projector to replace many image patch tokens with a few CLS embeddings while **keeping the LLM frozen**.

For each image view, its job is to extract the CLS token, project it, and insert it into the text embedding sequence at placeholder positions (vision module discussed earlier).

In [4]:
class PretrainVisionModel(pl.LightningModule):
    """
    Trains LoRA modules on the vision tower + projector to distill
    sub-image embeddings into a [CLS]-style compact representation.
    """
    def __init__(self, model, processor, tokenizer, args):
        super().__init__()
        self.model = model
        self.processor = processor
        self.tokenizer = tokenizer
        self.args = args

        self.data_collator = DataCollator(
            processor,
            tokenizer,
            max_length=args.max_length,
            prompt_template=PROMPT_TEMPLATE)
        self.save_hyperparameters(ignore=['model', 'processor', 'tokenizer'])

        # Running sums for validation
        self.val_loss_sum = 0.0
        self.val_token_count = 0

    def forward(self, input_ids, attention_mask, images, image_token_mask, labels=None):
        device = next(self.model.parameters()).device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        image_token_mask = image_token_mask.to(device)
        if labels is not None:
            labels = labels.to(device)

        core = get_llava_core(self.model)
        inputs_embeds = self.model.get_input_embeddings()(input_ids) # get token embeddings

        if images is not None:
            images = images.to(device, dtype=torch.float16)
            B, num_views, C, H, W = images.shape
            images_reshaped = images.view(B * num_views, C, H, W)             # Flatten image batch
            vision_outputs = core.vision_tower(images_reshaped)             # Vision encoder
            cls_states = vision_outputs.last_hidden_state[:, 0, :].view(B, num_views, -1)             # [CLS] states from each view
            cls_states = core.multi_modal_projector(cls_states)             # Project to LM hidden size

            # Replace placeholder token embeddings with projected CLS states
            for b_idx in range(B):
                positions = torch.nonzero(image_token_mask[b_idx], as_tuple=False).squeeze(-1)
                pos_count = min(len(positions), num_views)
                for i in range(pos_count):
                    col = positions[i].item()
                    inputs_embeds[b_idx, col, :] = cls_states[b_idx, i, :]


        # call the full model wrapper so we get logits + loss
        outputs = self.model(input_ids=None,attention_mask=attention_mask,inputs_embeds=inputs_embeds,labels=labels)
        return outputs

    def training_step(self, batch, batch_idx):
        inputs = self.data_collator(batch)
        outputs = self(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            images=inputs['images'],
            image_token_mask=inputs['image_token_mask'],
            labels=inputs['labels'])
        loss = outputs.loss
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True,
                 batch_size=len(batch))
        return loss

    def validation_step(self, batch, batch_idx):
        inputs = self.data_collator(batch)
        outputs = self(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            images=inputs['images'],
            image_token_mask=inputs['image_token_mask'],
            labels=inputs['labels']
        )
        val_loss = outputs.loss
        num_tokens = (inputs['labels'] != -100).sum().item()

        self.val_loss_sum += val_loss.item() * num_tokens
        self.val_token_count += num_tokens
        return val_loss

    def on_validation_epoch_end(self):
        if self.val_token_count > 0:
            avg_val_loss = self.val_loss_sum / self.val_token_count
        else:
            avg_val_loss = float('inf')
        ppl = math.exp(avg_val_loss) if avg_val_loss < 20 else float('inf')

        self.log('val_loss', avg_val_loss, prog_bar=True)
        self.log('val_perplexity', ppl, prog_bar=True)

        # Save metrics
        with open(os.path.join(self.args.output_dir, f"val_metrics_epoch_{self.current_epoch+1}.txt"), "w") as f:
            f.write(f"Val Loss: {avg_val_loss}\nVal PPL: {ppl}\n")

        self.val_loss_sum = 0.0
        self.val_token_count = 0

    def configure_optimizers(self):
        # Only optimize parameters that require grad (LoRA on vision tower + projector)
        params_to_optimize = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(
            params_to_optimize,
            lr=self.args.lr,
            weight_decay=self.args.weight_decay)
        return optimizer

## Recommendation Fine-Tuning

This is the second step of the two-part pipeline.  It trains the model to take a conversation and 10 candidate items (with images) and select the correct item ID, using the distilled visual features from Visual Knowledge Self-Distillation step above.

In [ ]:
# Here we formats the input into a structured prompt so the model
def build_prompt(conversation_text, candidates_info):
    """
    Always uses images in the prompt.
    """
    prompt = (
        "You are an AI assistant specialized in providing personalized product recommendations based on user conversations. "
        "You are given a conversation between a user seeking recommendation (denoted by <submission>) and other users providing comments (denoted by <comment>). "
        "You are also given a set of candidate products with their IDs, titles and images formatted as \"ID: title\" followed by an image. "
        "Among the candidates, recommend the most relevant product to the seeker. "
        "Only reply with its ID, and don't say anything else.\n\n"
        f"Conversation:\n{conversation_text}\n\n"
        "Candidates:\n"
    )
    for candidate in candidates_info:
        cid = candidate['id']
        title = candidate['title']
        prompt += f"{cid}: {title}\n"
        prompt += "".join(IMAGE_TOKENS) + "\n"

    prompt += "\nAssistant:"
    return prompt


This is the core of prompt tuning. The LLaVAModel fine‑tunes a pretrained LLaVA (LLaVA-v1.6) model for recommendation by applying LoRA to the language model side while **keeping the vision module frozen**. 

It uses 5 compact CLS embeddings from the prior distillation step to represent product images, and trains the model to output the most relevant product ID given a conversation context and a set of 10 candidates.

In [ ]:
class LLaVAModel(pl.LightningModule):
    """
    Model for recommendation prompt tuning:
    - Applies LoRA to the language model side.
    - Always uses images in the prompt.
    """
    def __init__(self, model, processor, tokenizer, args):
        super().__init__()
        self.model = model
        self.processor = processor
        self.tokenizer = tokenizer
        self.args = args
        self.save_hyperparameters(ignore=['model', 'processor', 'tokenizer'])

        self.data_collator = DataCollatorForLLaVA(processor, tokenizer, max_length=args.max_length)
        self.test_results = []

    def forward(self, input_ids, attention_mask, images, image_token_mask, images_per_sample_lengths, labels=None):
      device = next(self.model.parameters()).device
      input_ids = input_ids.to(device)
      attention_mask = attention_mask.to(device)
      image_token_mask = image_token_mask.to(device)
      if labels is not None:
          labels = labels.to(device)

      core = get_llava_core(self.model)

      # text embeddings from full wrapped model
      inputs_embeds = self.model.get_input_embeddings()(input_ids)

      if images is not None:
          images = images.to(device, dtype=torch.float16)

          B_text = input_ids.size(0)
          B_prime = images.shape[0]           # total candidate images across batch
          num_views = images.shape[1]         # should be 5
          C, H, W = images.shape[2], images.shape[3], images.shape[4]

          # flatten for vision tower
          images_reshaped = images.view(B_prime * num_views, C, H, W)

          with torch.no_grad():
              vision_outputs = core.vision_tower(images_reshaped)

          # take CLS per sub-image
          cls_states = vision_outputs.last_hidden_state[:, 0, :]   # (B_prime * 5, hidden)
          cls_states = cls_states.view(B_prime, num_views, -1)     # (B_prime, 5, hidden)

          candidate_count = B_prime // B_text
          cls_states = cls_states.view(B_text, candidate_count, num_views, -1)

          # project to LM hidden size
          cls_states = core.multi_modal_projector(cls_states)

          # replace placeholder image-token embeddings
          for b_idx in range(B_text):
              image_positions = torch.nonzero(image_token_mask[b_idx], as_tuple=False).squeeze(-1)
              if image_positions.numel() == 0:
                  continue

              needed_tokens = candidate_count * num_views
              pos_count = min(len(image_positions), needed_tokens)

              for c in range(candidate_count):
                  offset_c = c * num_views
                  for i in range(num_views):
                      idx_token = offset_c + i
                      if idx_token >= pos_count:
                          break
                      col = image_positions[idx_token].item()
                      inputs_embeds[b_idx, col, :] = cls_states[b_idx, c, i, :]

      # call full wrapped model so outputs.loss exists
      outputs = self.model(
          input_ids=None,
          attention_mask=attention_mask,
          inputs_embeds=inputs_embeds,
          labels=labels)
      return outputs

    def training_step(self, batch, batch_idx):
        inputs = self.data_collator(batch)
        outputs = self(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            images=inputs['images'],
            image_token_mask=inputs['image_token_mask'],
            images_per_sample_lengths=inputs['images_per_sample_lengths'],
            labels=inputs['labels'], )
        loss = outputs.loss
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, batch_size=len(batch))
        return loss

    def validation_step(self, batch, batch_idx):
        inputs = self.data_collator(batch)
        outputs = self(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            images=inputs['images'],
            image_token_mask=inputs['image_token_mask'],
            images_per_sample_lengths=inputs['images_per_sample_lengths'],
            labels=inputs['labels'], )
        val_loss = outputs.loss
        self.log('val_loss', val_loss, on_epoch=True, prog_bar=True, batch_size=len(batch))
        return {'val_loss': val_loss}

    
    def test_step(self, batch, batch_idx):
      with torch.no_grad():
          inputs = self.data_collator(batch)
          device = next(self.model.parameters()).device
          input_ids = inputs['input_ids'].to(device)
          attention_mask = inputs['attention_mask'].to(device)
          image_token_mask = inputs['image_token_mask'].to(device)
          images = inputs['images']
          core = get_llava_core(self.model)
          
          # text embeddings from full wrapped model
          inputs_embeds = self.model.get_input_embeddings()(input_ids)

          if images is not None:
              images = images.to(device, dtype=torch.float16)

              B_text = input_ids.size(0)
              B_prime = images.shape[0]
              num_views = images.shape[1]
              C, H, W = images.shape[2], images.shape[3], images.shape[4]
              
              images_reshaped = images.view(B_prime * num_views, C, H, W)
              vision_outputs = core.vision_tower(images_reshaped)
              cls_states = vision_outputs.last_hidden_state[:, 0, :]
              cls_states = cls_states.view(B_prime, num_views, -1)
              
              candidate_count = B_prime // B_text
              cls_states = cls_states.view(B_text, candidate_count, num_views, -1)
              cls_states = core.multi_modal_projector(cls_states)

              for b_idx in range(B_text):
                  image_positions = torch.nonzero(image_token_mask[b_idx], as_tuple=False).squeeze(-1)
                  if image_positions.numel() == 0:
                      continue

                  needed_tokens = candidate_count * num_views
                  pos_count = min(len(image_positions), needed_tokens)

                  for c in range(candidate_count):
                      offset_c = c * num_views
                      for i in range(num_views):
                          idx_token = offset_c + i
                          if idx_token >= pos_count:
                              break
                          col = image_positions[idx_token].item()
                          inputs_embeds[b_idx, col, :] = cls_states[b_idx, c, i, :]

          # generate from full wrapped model, not language_model directly
          generated_ids = self.model.generate(
              input_ids=None,
              inputs_embeds=inputs_embeds,
              attention_mask=attention_mask,
              max_new_tokens=10,
              num_beams=1,
              do_sample=False,
              pad_token_id=self.tokenizer.pad_token_id, )

          generated_texts = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
          recommended_ids = []
          
          for txt in generated_texts:
              match = re.findall(r'\bB[A-Z0-9]{9}\b', txt.strip())
              recommended_ids.append(match[0][:10] if match else None)

          gt_items_list = [b['gt_items'] for b in batch]
          entry_idxs = [b['entry_idx'] for b in batch]

          for i in range(len(batch)):
              self.test_results.append({
                  'entry_idx': entry_idxs[i],
                  'recommended_id': recommended_ids[i],
                  'response': generated_texts[i] })

          recall = evaluate_recall_at_k(recommended_ids, gt_items_list, k=1)
          self.log('test_recall', recall, on_step=False, on_epoch=True, prog_bar=True, batch_size=len(batch))
          return {'test_recall': recall}

    def configure_optimizers(self):
        params = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW( params,lr=self.args.lr, weight_decay=self.args.weight_decay)
        return optimizer

## Results

The results of the 3 sub-training sets of 1, 2, and all images are summarized below alongside the baseline of text-only embeddings:

| Name |Learning Rate |  Weight_Decay |Batch Size | No. of Epochs | Recall@1 | Validity@1 |Recall@1/Validity@1 (Final Score)|
| --- | --- | --- | --- |--- |--- |--- |--- |
| Baseline, text-only| 0.00005| 0.00001 | 2| 2|0.0378|0.9866|<font color="red">**0.0383**</font>
| ~2500 images (single-distillation)| 0.00005| 0.00001 | 2| 2|0.1662|0.9651|<font color="red">**0.1722**</font>
| ~5600 images (dual-distillation)| 0.00005| 0.00001 | 2| 2|0.2469|0.9973|<font color="red">**0.2476**</font>
| ~18000 images (full-scale-distillation)| 0.00005| 0.00001 | 2| 2|0.2443|0.9919|<font color="red">**0.2463**</font>

It is evident from the results of the experiments that the dual-distillation trial performed the best with a Recall@1/Validity@1 score of 0.2476. What is interesting to note is that although the model's performance improved from the single-distillation trial to the dual-distillation trial, increasing from 2 to all images during training from the second to the third trial did not improve the metrics, and ironically decreased them slightly, indicating that scaling the number of images past 2 may not be necessary as the model was able to learn and extract the most visual meaning it could from just 2 images per item.

# Conclusion and Future Direction

Future work can focus on improving the efficiency of the overall pipeline. Even though the visual knowledge distillation step reduces the number of image tokens, the model is still expensive to train.  Methods like quantization, more efficient LoRA settings, or caching the visual embeddings can help reduce runtime and memory usage. Using larger backbone models could also improve performance, but this comes with a clear trade-off in computation cost, so it needs to be balanced carefully.

Stepping back from algorithm efficiency point, we can also incorporate user history for better personalization. Right now, the model only uses the current conversation, which means it cannot capture long-term preferences. In practice, users often have consistent tastes, so adding information like past purchases, browsing history, or learned user embeddings could improve recommendation quality.

Lastly, we can extend the types of information used by the model. Currently, the system mainly relies on images and titles, but many products require more context. Adding features such as product descriptions, reviews, or structured attributes can provide additional signals that are not purely visual. This is especially useful for items where functionality matters more than appearance.

# References:

[1] Liu, Y., Wang, Y., Sun, L., & Yu, P. S. (2024). Multimodal Recommendation with Large Vision-Language Models. arXiv:2402.09142.

[2] Wei, Y., Zou, J., Guo, W., Wang, G., Xu, X., & Yang, Y. (2025). MSCRS: Multi-modal Semantic Graph Prompt Learning Framework for Conversational Recommender Systems. arXiv:2504.10921.

[3] Jeon, H., Koide, S., Wang, Y., He, Z., & McAuley, J. (2025). LaViC: Adapting Large Vision-Language Models to Visually-Aware Conversational Recommendation. arXiv:2503.23312.

[4] Hugging Face. llava-hf/llava-v1.6-mistral-7b-hf. Available at: https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf
